<a href="https://colab.research.google.com/github/bvsvntv/ai-ml-learning/blob/main/notebooks/lang-graph/basic_stm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.1/588.1 kB 15.5 MB/s  0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.7.4
    Uninstalling langchain-core-1.7.4:
      Successfully uninstalled langchain-core-1.7.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [langchain-openai]


In [14]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langchain_openai import ChatOpenAI
from google.colab import userdata
from langgraph.checkpoint.memory import InMemorySaver

In [3]:
api_key = userdata.get("OPENAI_API_KEY")
model = ChatOpenAI(api_key=api_key, model="gpt-6-luna")

In [23]:
def call_model(state: MessagesState):
  response = model.invoke(state["messages"])
  return {"messages": [response]}

In [24]:
graph = StateGraph(MessagesState)
graph.add_node("call_model", call_model)

graph.add_edge(START, "call_model")
graph.add_edge("call_model", END)

In [25]:
workflow = graph.compile()

No `thread_id` and `checkpointer` is used yet.

In [12]:
workflow.invoke({"messages": [{"role": "user", "content": "Hi, My name is James Bond."}]})

{'messages': [HumanMessage(content='Hi, My name is James Bond.', additional_kwargs={}, response_metadata={}, id='5c0dda75-1424-4826-b926-ac037c18f88c'),
  AIMessage(content='Hi, James! Nice to meet you. How can I help?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 14, 'total_tokens': 65, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 28, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-Ektip4sTE2lmvc5UTm0idxR5Qg0Va', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1e528-5943-77b1-89e4-347585c6adbc-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 51, 'total_tokens': 65, 'input_token_details': {'audio': 0, 'c

In [13]:
workflow.invoke({"messages": [{"role": "user", "content": "What is my name?"}]})

{'messages': [HumanMessage(content='What is my name?', additional_kwargs={}, response_metadata={}, id='c795416c-0f5c-47a8-b282-70cd155882c3'),
  AIMessage(content='I don’t know—you haven’t told me your name.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 11, 'total_tokens': 57, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 25, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EktjKoeUR4rRFs833LFVKzW0g13Vr', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1e528-d660-7394-b046-7b578473180a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 11, 'output_tokens': 46, 'total_tokens': 57, 'input_token_details': {'audio': 0, 'cache_read':

STM is not implemented yet, so no information is stored in the graph.

Let's implement STM, with in memory checkpoint

In [27]:
checkpointer = InMemorySaver()
workflow_with_stm = graph.compile(checkpointer=checkpointer)

In [30]:
config1 = {"configurable": {"thread_id": "thread-1"}}
config2 = {"configurable": {"thread_id": "thread-2"}}

In [31]:
workflow_with_stm.invoke({"messages": [{"role": "user", "content": "Hi, My name is James Bond."}]}, config=config1)

{'messages': [HumanMessage(content='Hi, My name is James Bond.', additional_kwargs={}, response_metadata={}, id='691953c0-739b-4d83-9060-01841df1664d'),
  AIMessage(content='Hello, James Bond. How can I help you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 62, 'prompt_tokens': 14, 'total_tokens': 76, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 41, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-Ekts1UyzToutWylstCebCcPjsjW3D', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1e531-120d-7685-965e-d81086346108-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 62, 'total_tokens': 76, 'input_token_details': {'audio': 0, 'c

In [32]:
workflow_with_stm.invoke({"messages": [{"role": "user", "content": "What is my name?"}]}, config=config1)

{'messages': [HumanMessage(content='Hi, My name is James Bond.', additional_kwargs={}, response_metadata={}, id='691953c0-739b-4d83-9060-01841df1664d'),
  AIMessage(content='Hello, James Bond. How can I help you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 62, 'prompt_tokens': 14, 'total_tokens': 76, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 41, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-Ekts1UyzToutWylstCebCcPjsjW3D', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1e531-120d-7685-965e-d81086346108-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 62, 'total_tokens': 76, 'input_token_details': {'audio': 0, 'c

In [34]:
graph_state = workflow_with_stm.get_state(config=config1)
values = graph_state.values
for m in values.get("messages", []):
  print("-", type(m).__name__, ":", m.content)


- HumanMessage : Hi, My name is James Bond.
- AIMessage : Hello, James Bond. How can I help you today?
- HumanMessage : What is my name?
- AIMessage : You told me your name is James Bond.


In [35]:
workflow_with_stm.invoke({"messages": [{"role": "user", "content": "What is my name?"}]}, config=config2)

{'messages': [HumanMessage(content='What is my name?', additional_kwargs={}, response_metadata={}, id='163d8b68-11d1-496a-8ba0-3142ca38a257'),
  AIMessage(content='I don’t know your name—you haven’t told me.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 11, 'total_tokens': 43, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 11, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EktvA7URPKCW4OMRX5LzM5PR98AMb', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1e534-0b64-717a-80f5-d7ab582952c0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 11, 'output_tokens': 32, 'total_tokens': 43, 'input_token_details': {'audio': 0, 'cache_read':

In [37]:
graph_state_2 = workflow_with_stm.get_state(config=config2)
values = graph_state_2.values
for m in values.get("messages", []):
  print("-", type(m).__name__, ":", m.content)

- HumanMessage : What is my name?
- AIMessage : I don’t know your name—you haven’t told me.
